# Exp-1 temporal search: connection-safe retry

This notebook retries each existing `rank=None` record exactly once, then force-reruns benchmark rows 70–78. It keeps GPT-4o planning and SigLIP2 semantic search fixed, checkpoints every request, and preserves valid records unless a row is explicitly force-rerun.

In [8]:
from pathlib import Path
import sys

# Locate the repository from either its root or the notebook directory.
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data').exists() and (path / 'notebooks').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the Multimodal-Retrieval workspace.')
RUNNER = ROOT / 'notebooks/experiments/exp1_temporal_search/run_exp1_temporal_search.py'
OUTPUT_DIR = ROOT / 'data/experiments/results/exp1_temporal_search_v2'
assert RUNNER.exists() and OUTPUT_DIR.exists()
print(f'Runner: {RUNNER}')
print(f'Output: {OUTPUT_DIR}')

Runner: d:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\experiments\exp1_temporal_search\run_exp1_temporal_search.py
Output: d:\University\Projects\Individual projects\Multimodal-Retrieval\data\experiments\results\exp1_temporal_search_v2


In [9]:
# Inspect the records to be retried. A rank=None may be a genuine miss,
# but this retry verifies it was not caused by a transient model outage.
import json

for method in ('ats', 'vortex', 'dev'):
    checkpoint = OUTPUT_DIR / method / 'checkpoint.jsonl'
    records = [json.loads(line) for line in checkpoint.read_text(encoding='utf-8').splitlines() if line]
    retry_ids = [row['query_id'] for row in records if row.get('rank') is None]
    print(f'{method}: {len(retry_ids)} rank=None -> {retry_ids}')

ats: 30 rank=None -> ['query-p1-5-kis', 'query-p1-6-kis', 'query-p1-8-kis', 'query-p1-10-kis', 'query-p1-13-kis', 'query-p1-25-kis', 'query-p2-4-kis', 'query-p2-5-kis', 'query-p2-6-kis', 'query-p2-14-kis', 'query-p2-26-kis', 'query-p2-29-qa', 'query-p3-4-kis', 'query-p3-11-kis', 'query-p3-12-kis', 'query-p3-16-kis', 'query-p3-17-kis', 'query-p3-20-kis', 'query-p3-24-qa', 'query-p3-26-kis', 'query-p3-28-kis', 'query-p3-29-kis', 'query-p3-31-kis', 'query-p3-32-kis', 'query-p3-33-kis', 'query-p1-6-kis', 'query-p1-6-kis', 'query-p1-8-kis', 'query-p1-10-kis', 'query-p1-13-kis']
vortex: 15 rank=None -> ['query-p1-10-kis', 'query-p2-4-kis', 'query-p2-5-kis', 'query-p2-26-kis', 'query-p2-29-qa', 'query-p3-11-kis', 'query-p3-12-kis', 'query-p3-16-kis', 'query-p3-17-kis', 'query-p3-24-qa', 'query-p3-28-kis', 'query-p3-29-kis', 'query-p3-31-kis', 'query-p3-32-kis', 'query-p1-10-kis']
dev: 11 rank=None -> ['query-p1-10-kis', 'query-p2-26-kis', 'query-p2-29-qa', 'query-p3-12-kis', 'query-p3-17-kis'

In [10]:
# Stream output line-by-line in Jupyter and retain a disk log.
from subprocess import PIPE, STDOUT, Popen
from datetime import datetime

def run_with_live_log(*arguments):
    log_dir = OUTPUT_DIR / 'retry_logs'
    log_dir.mkdir(exist_ok=True)
    log_path = log_dir / f"retry_{datetime.now():%Y%m%d_%H%M%S}.log"
    command = [sys.executable, str(RUNNER), *map(str, arguments)]
    print('Running:', ' '.join(command))
    print('Live log:', log_path)
    with log_path.open('w', encoding='utf-8') as log_file:
        process = Popen(command, stdout=PIPE, stderr=STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
    if process.wait() != 0:
        raise RuntimeError(f'Runner failed; inspect {log_path}')
    return log_path

# Retry only rank=None records once. Valid ranked records are skipped.
retry_log = run_with_live_log('--output-dir', OUTPUT_DIR, '--retry-rank-none')

Running: c:\Python314\python.exe d:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\experiments\exp1_temporal_search\run_exp1_temporal_search.py --output-dir d:\University\Projects\Individual projects\Multimodal-Retrieval\data\experiments\results\exp1_temporal_search_v2 --retry-rank-none
Live log: d:\University\Projects\Individual projects\Multimodal-Retrieval\data\experiments\results\exp1_temporal_search_v2\retry_logs\retry_20260918_233158.log
PLAN 5/78 query-p1-6-kis: GPT-4o cached in 16152 ms
ATS query-p1-6-kis: rank=9 error=
PLAN 7/78 query-p1-8-kis: GPT-4o cached in 70 ms
ATS query-p1-8-kis: rank=None error=
PLAN 9/78 query-p1-10-kis: GPT-4o cached in 45 ms
ATS query-p1-10-kis: rank=None error=
Vortex query-p1-10-kis: rank=None error=
DEV query-p1-10-kis: rank=None error=
PLAN 12/78 query-p1-13-kis: GPT-4o cached in 45 ms
ATS query-p1-13-kis: rank=None error=
PLAN 21/78 query-p1-25-kis: GPT-4o cached in 55 ms
ATS query-p1-25-kis: rank=None error=
PLAN 25/78 

In [ ]:
# Force a fair post-recovery rerun of benchmark positions 70–78 (inclusive).
# These are p3-27-qa through p3-36-kis in the supplied CSV.
ROWS_70_78 = ','.join([
    'query-p3-27-qa', 'query-p3-28-kis', 'query-p3-29-kis',
    'query-p3-30-kis', 'query-p3-31-kis', 'query-p3-32-kis',
    'query-p3-33-kis', 'query-p3-35-qa', 'query-p3-36-kis',
])
rows_70_78_log = run_with_live_log('--output-dir', OUTPUT_DIR, '--query-ids', ROWS_70_78, '--force')

python: can't open file 'd:\\University\\Projects\\Individual': [Errno 2] No such file or directory


In [ ]:
# Report remaining rank=None records and the regenerated full metrics.
for method in ('ats', 'vortex', 'dev'):
    checkpoint = OUTPUT_DIR / method / 'checkpoint.jsonl'
    latest = {}
    for line in checkpoint.read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        latest[row['query_id']] = row
    remaining = [row['query_id'] for row in latest.values() if row.get('rank') is None]
    print(f'{method}: remaining rank=None = {len(remaining)}')

print((OUTPUT_DIR / 'temporal_search_summary.csv').read_text(encoding='utf-8-sig'))